# Static XAS — multi-run comparison
Compare three static-XAS runs with the same processed H5 schema used by the single-run notebooks.

Runs included:
- `58780` → `no_sample`
- `58793` → `1mmol`
- `58794` → `5mmol`

For each run, this notebook builds two XAS views using the same logic as the existing notebooks:
1. `XAS` vs nominal photon energy.
2. `XAS` vs shot-resolved photon energy proxy from the VLS centre-of-mass, rounded to the nearest Gotthard pixel.

Only two final comparison figures are produced: one for the nominal-energy axis and one for the shot-resolved proxy axis.


In [ ]:
import sys
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

cwd = Path.cwd().resolve()
repo_root = None
for p in [cwd] + list(cwd.parents):
    if (p / 'analysis' / 'scripts').exists():
        repo_root = p
        break
if repo_root is None:
    raise RuntimeError('Could not locate repository root.')
sys.path.insert(0, str(repo_root / 'analysis' / 'scripts'))
import config as path_config


## Config


In [ ]:
RUNS = [
    {'run_no': 58780, 'label': 'no_sample'},
    {'run_no': 58793, 'label': '1mmol'},
    {'run_no': 58794, 'label': '5mmol'},
]

XAS_STATIC_DIR = Path(path_config.COMBINED_DIR).parent / 'xas_static'

# Shared GMD bin edges used for all runs so the comparison is on the same intensity axis.
GMD_EDGES = np.array([0.0, 1.6, 3.2, 6.4, 12.8])

# Proxy-only shot filter. The nominal-energy plot uses all finite shots, matching the single-run notebook.
VLS_INTENSITY_THRESHOLD = 1000

for run in RUNS:
    run['path'] = XAS_STATIC_DIR / f"run{run['run_no']}_static_xas.h5"

print(f'XAS static dir: {XAS_STATIC_DIR}')
for run in RUNS:
    print(f"run {run['run_no']} ({run['label']}): {run['path']}")


## Helpers

The helpers below keep the binning logic aligned with the existing nominal-energy and shot-resolved notebooks, but package it for multiple runs.


In [ ]:
def load_static_xas_run(path):
    with h5py.File(path, 'r') as f:
        return {
            'vls': f['vls'][...],
            'gmd': f['gmd'][...],
            'n_shots': f['n_shots'][...],
            'nominal_energies': f['nominal_energies'][...],
            'vls_pixels': f['vls_pixels'][...],
            'attrs': dict(f.attrs),
        }


def bin_nominal_energy_run(data, gmd_edges):
    vls = data['vls']
    gmd = data['gmd']
    n_shots = data['n_shots']
    nominal_energies = data['nominal_energies']

    n_energy, _, n_pixels = vls.shape
    n_gmd = len(gmd_edges) - 1

    a_sum = np.zeros((n_energy, n_gmd, n_pixels), dtype=np.float64)
    g_sum = np.zeros((n_energy, n_gmd), dtype=np.float64)
    n_per_bin = np.zeros((n_energy, n_gmd), dtype=np.int64)

    for ie in range(n_energy):
        n = int(n_shots[ie])
        g = gmd[ie, :n]
        a = vls[ie, :n, :]

        finite = np.isfinite(g) & np.all(np.isfinite(a), axis=1)
        if not np.any(finite):
            continue

        g = g[finite]
        a = a[finite]
        bins = np.digitize(g, gmd_edges) - 1
        in_range = (bins >= 0) & (bins < n_gmd)
        if not np.any(in_range):
            continue

        g = g[in_range]
        a = a[in_range]
        bins = bins[in_range]

        np.add.at(n_per_bin[ie], bins, 1)
        np.add.at(g_sum[ie], bins, g)
        np.add.at(a_sum[ie], bins, a)

    safe_n = np.where(n_per_bin > 0, n_per_bin, 1).astype(np.float64)
    g_mean = np.where(n_per_bin > 0, g_sum / safe_n, np.nan)
    a_mean = np.where(n_per_bin[:, :, None] > 0, a_sum / safe_n[:, :, None], np.nan)
    vls_sum = np.nansum(a_mean, axis=2)

    with np.errstate(divide='ignore', invalid='ignore'):
        xas = g_mean / vls_sum

    return {
        'x': nominal_energies,
        'xas': xas,
        'n_per_bin': n_per_bin,
    }


def prepare_proxy_shots(data, vls_intensity_threshold=None):
    vls = data['vls']
    gmd = data['gmd']
    nominal_energies = data['nominal_energies']
    n_pixels = vls.shape[-1]

    gmd_flat = gmd.reshape(-1)
    vls_flat = vls.reshape(-1, n_pixels)
    nominal_energy_flat = np.repeat(nominal_energies, vls.shape[1])

    ok = np.isfinite(gmd_flat) & np.all(np.isfinite(vls_flat), axis=1)
    g_valid = gmd_flat[ok]
    A_valid = vls_flat[ok]
    nominal_energy_valid = nominal_energy_flat[ok]

    peak_vls = np.nanmax(A_valid, axis=1)
    if vls_intensity_threshold is not None:
        keep_peak = peak_vls >= vls_intensity_threshold
        g_valid = g_valid[keep_peak]
        A_valid = A_valid[keep_peak]
        nominal_energy_valid = nominal_energy_valid[keep_peak]

    A_pos = np.maximum(A_valid, 0.0)
    pix = np.arange(n_pixels, dtype=np.float64)
    A_sum_per_shot = A_pos.sum(axis=1)
    nonzero = A_sum_per_shot > 0

    com_float = np.full(g_valid.size, np.nan, dtype=np.float64)
    com_float[nonzero] = (A_pos[nonzero] @ pix) / A_sum_per_shot[nonzero]

    valid_com = np.isfinite(com_float)
    g_valid = g_valid[valid_com]
    A_valid = A_valid[valid_com]
    nominal_energy_valid = nominal_energy_valid[valid_com]
    proxy_pixel = np.round(com_float[valid_com]).astype(np.int64)

    return {
        'g_valid': g_valid,
        'A_valid': A_valid,
        'nominal_energy_valid': nominal_energy_valid,
        'proxy_pixel': proxy_pixel,
    }


def bin_proxy_energy(prepared, vls_pixels, gmd_edges):
    N_GMD = len(gmd_edges) - 1

    A_valid = prepared['A_valid']
    g_valid = prepared['g_valid']
    proxy_pixel = prepared['proxy_pixel']
    n_pixels = A_valid.shape[1]

    proxy_energies = np.sort(np.unique(proxy_pixel))
    pixel_to_idx = {int(p): i for i, p in enumerate(proxy_energies)}
    proxy_idx = np.array([pixel_to_idx[int(p)] for p in proxy_pixel], dtype=np.int64)
    N_proxy_E = len(proxy_energies)

    A_sum = np.zeros((N_proxy_E, N_GMD, n_pixels), dtype=np.float64)
    G_sum = np.zeros((N_proxy_E, N_GMD), dtype=np.float64)
    n_per_bin = np.zeros((N_proxy_E, N_GMD), dtype=np.int64)

    gmd_bins = np.digitize(g_valid, gmd_edges) - 1
    in_range = (gmd_bins >= 0) & (gmd_bins < N_GMD)

    pe = proxy_idx[in_range]
    gb = gmd_bins[in_range]
    G_ir = g_valid[in_range]
    A_ir = A_valid[in_range]

    np.add.at(n_per_bin, (pe, gb), 1)
    np.add.at(G_sum, (pe, gb), G_ir)
    np.add.at(A_sum, (pe, gb), A_ir)

    safe_n = np.where(n_per_bin > 0, n_per_bin, 1).astype(np.float64)
    G_mean = np.where(n_per_bin > 0, G_sum / safe_n, np.nan)
    A_mean = np.where(n_per_bin[:, :, None] > 0, A_sum / safe_n[:, :, None], np.nan)
    vls_sum = np.nansum(A_mean, axis=2)

    with np.errstate(divide='ignore', invalid='ignore'):
        xas = G_mean / vls_sum

    proxy_pixels_abs = vls_pixels[proxy_energies].astype(np.float64)

    return {
        'x': proxy_pixels_abs,
        'x_rel': proxy_energies,
        'xas': xas,
        'n_per_bin': n_per_bin,
    }


def plot_xas_panels(results_by_run, axis_key, gmd_edges, figure_title, x_label):
    N_GMD = len(gmd_edges) - 1
    cmap = plt.get_cmap('viridis')

    fig, axes = plt.subplots(1, len(results_by_run), figsize=(6.0 * len(results_by_run), 4.6), sharey=True)
    axes = np.atleast_1d(axes)

    for ax, run_result in zip(axes, results_by_run):
        x = run_result[axis_key]['x']
        xas = run_result[axis_key]['xas']

        for g in range(N_GMD):
            color = cmap(g / max(N_GMD - 1, 1))
            ax.plot(
                x,
                xas[:, g],
                lw=1.8,
                color=color,
                label=f'[{gmd_edges[g]:.2f}, {gmd_edges[g + 1]:.2f}) uJ',
            )

        ax.set_title(f"run {run_result['run_no']} — {run_result['label']}")
        ax.set_xlabel(x_label)
        ax.grid(alpha=0.3)

    axes[0].set_ylabel('XAS = <GMD> / sum_pix(<VLS>)')
    axes[-1].legend(title='GMD bin', fontsize=8, title_fontsize=9, loc='best')
    fig.suptitle(figure_title, fontsize=14)
    fig.tight_layout()
    plt.show()


## Load and process runs
The nominal-energy branch uses the raw per-energy shot bins, just like the single-run notebook. The proxy branch applies the VLS peak filter and COM validity check.


In [ ]:
missing = [str(run['path']) for run in RUNS if not run['path'].exists()]
if missing:
    raise FileNotFoundError(
        'Missing processed static-XAS files. Build them first, then rerun this cell:\n' + '\n'.join(missing)
    )

results_by_run = []
for run in RUNS:
    data = load_static_xas_run(run['path'])
    nominal = bin_nominal_energy_run(data, GMD_EDGES)
    proxy_prepared = prepare_proxy_shots(data, vls_intensity_threshold=VLS_INTENSITY_THRESHOLD)
    proxy = bin_proxy_energy(proxy_prepared, data['vls_pixels'], GMD_EDGES)

    results_by_run.append({
        'run_no': run['run_no'],
        'label': run['label'],
        'path': run['path'],
        'attrs': data['attrs'],
        'vls_pixels': data['vls_pixels'],
        'nominal': nominal,
        'proxy': proxy,
        'n_valid_shots': len(proxy_prepared['g_valid']),
    })

for run_result in results_by_run:
    print(
        f"run {run_result['run_no']} ({run_result['label']}): "
        f"proxy-valid shots={run_result['n_valid_shots']}, "
        f"nominal bins={len(run_result['nominal']['x'])}, "
        f"proxy bins={len(run_result['proxy']['x'])}, "
        f"proxy pixel range={run_result['proxy']['x'][0]:.0f}..{run_result['proxy']['x'][-1]:.0f}"
    )


## XAS vs nominal photon energy
One panel per run. Within each panel, the lines correspond to the shared GMD bins.


In [ ]:
plot_xas_panels(
    results_by_run=results_by_run,
    axis_key='nominal',
    gmd_edges=GMD_EDGES,
    figure_title='Static XAS comparison: nominal photon energy',
    x_label='nominal photon energy (eV)',
)


## XAS vs shot-resolved photon energy
The x-axis is the rounded VLS centre-of-mass converted to absolute Gotthard pixel index.


In [ ]:
plot_xas_panels(
    results_by_run=results_by_run,
    axis_key='proxy',
    gmd_edges=GMD_EDGES,
    figure_title='Static XAS comparison: shot-resolved photon energy proxy',
    x_label='shot-resolved photon energy proxy (Gotthard pixel)',
)
